# FirePath AI on SageMaker

**Instance:** `ml.g4dn.xlarge` (T4 GPU), volume 50 GB. **Kernel:** `conda_pytorch_p310` (or the newest `conda_pytorch_*`).

Put your files in the JupyterLab file browser's top folder (`/home/ec2-user/SageMaker`, the only folder that survives a stop/start):
- `firepath.zip` (optional — the notebook clones from GitHub first)
- `D-Fire.zip`
- a `videos/` folder with your evaluation videos, `videos.csv` and `calib.csv` (step 4)

Small files: drag them into the file browser. Big files (D-Fire): upload to S3 and use the S3 line in step 2.

**Stop the instance when you're done** (SageMaker → Notebook instances → Stop). It bills every hour it's running, even idle.

In [1]:
import sys, os
BASE = '/home/ec2-user/SageMaker'
PY = sys.executable          # use the kernel's python for every command below
print('Python:', PY)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('CUDA available:', torch.cuda.is_available())

Python: /opt/conda/bin/python
name, memory.total [MiB]
Tesla T4, 15360 MiB
CUDA available: True


## 0. Get the project

In [ ]:
import glob, subprocess
PROJ = f'{BASE}/firepath'
if not os.path.exists(f'{PROJ}/requirements.txt'):
    r = subprocess.run(['git', 'clone', '-q', 'https://github.com/Aditi-Nandiraju/Firepath-CV.git', PROJ])
    if r.returncode != 0:
        if not os.path.exists(f'{BASE}/firepath.zip'):
            raise SystemExit('Clone failed and firepath.zip not found in ' + BASE)
        subprocess.run(['unzip', '-q', '-o', f'{BASE}/firepath.zip', '-d', BASE], check=True)
if not os.path.exists(f'{PROJ}/requirements.txt'):   # zip extracted under another name
    PROJ = os.path.dirname(sorted(glob.glob(f'{BASE}/**/requirements.txt', recursive=True), key=len)[0])
print('Project at', PROJ)
%cd $PROJ

In [ ]:
%pip install -q -r requirements.txt
# SageMaker's Amazon Linux lacks the GUI libraries the normal OpenCV needs; swap to the headless build if import fails
try:
    import cv2
except ImportError:
    !{PY} -m pip uninstall -y -q opencv-python
    %pip install -q opencv-python-headless
print('Restart the kernel once (Kernel → Restart) if cv2 still fails, then re-run from the top.')

## 1. Quick test of the simulation (placeholder numbers, NOT results)

In [ ]:
!{PY} -m firepath.experiments --scenarios 60 --skip-ablation --out results_demo
from IPython.display import Image, display
display(Image('results_demo/tradeoff.png'))

In [ ]:
!{PY} -m firepath.visualize --find-interesting --out results_demo/route_example.png
display(Image('results_demo/route_example.png'))

## 2. Get D-Fire
Either drag `D-Fire.zip` into the top folder, or upload it to S3 and fill in `S3_DFIRE`.

In [ ]:
S3_DFIRE = ''   # e.g. 's3://my-bucket/D-Fire.zip'  (leave empty if you uploaded the zip directly)
DFIRE = f'{BASE}/D-Fire'
if S3_DFIRE and not os.path.exists(f'{BASE}/D-Fire.zip'):
    !aws s3 cp $S3_DFIRE $BASE/D-Fire.zip
if not os.path.isdir(DFIRE):
    !unzip -q -o $BASE/D-Fire.zip -d $DFIRE
!ls $DFIRE

Check which class id is fire and which is smoke. The output tells you what to pass as `--names`. If `ls` shows a single sub-folder, set `DFIRE` to that sub-folder.

In [ ]:
!{PY} detection/check_dataset.py --root $DFIRE

## 3. Fine-tune YOLO (a few hours)
Runs in the background and logs to `train.log`, so it keeps going if you close the browser. Change `--names` if step 2 said so; try `EPOCHS = 10` first to test.

In [ ]:
EPOCHS = 50
!nohup {PY} detection/train_yolo.py --root $DFIRE --names smoke fire --epochs $EPOCHS --model yolov8n.pt > train.log 2>&1 &
print('Training started. Re-run the next cell to check progress.')

In [ ]:
# progress check — re-run any time (also after reopening JupyterLab)
!tail -n 5 train.log
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
print('Done:', os.path.exists('runs/firepath/train/weights/best.pt') and 'best.pt exists' or 'not yet')

Once training has finished (log shows the final results), keep a copy of the weights:

In [ ]:
!mkdir -p $BASE/saved && cp runs/firepath/train/weights/best.pt $BASE/saved/

## 4. Measure the detector's real errors on videos
Make a `videos/` folder in the top folder with your evaluation videos (NOT D-Fire training images) and `videos.csv` / `calib.csv` (see `configs/videos_template.csv`). Paths in the CSVs must be full paths on this machine, e.g. `/home/ec2-user/SageMaker/videos/sunset1.mp4`. Labels: `fire`, `smoke`, `none`, `confuser`.

In [ ]:
VID = f'{BASE}/videos'
!{PY} detection/measure_detector.py --weights runs/firepath/train/weights/best.pt \
    --videos $VID/videos.csv \
    --calib-videos $VID/calib.csv \
    --out configs/detector_errors_measured.json
!cp configs/detector_errors_measured.* $BASE/saved/

## 5. Choose λ on validation scenarios, then run the real experiments

In [ ]:
!{PY} -m firepath.experiments --errors configs/detector_errors_measured.json --seed0 100000 --scenarios 200 --skip-ablation --out results_val
display(Image('results_val/tradeoff.png'))

In [ ]:
!{PY} -m firepath.experiments --errors configs/detector_errors_measured.json --seed0 0 --scenarios 300 --out results_test
print(open('results_test/summary.md').read())
print(open('results_test/error_types.md').read())
display(Image('results_test/tradeoff.png'))
!cp -r results_test $BASE/saved/

## 6. Download your results
Right-click `saved.zip` in the file browser → Download. Optionally also copy to S3. Then **stop the instance**.

In [ ]:
!cd $BASE && zip -q -r saved.zip saved
S3_OUT = ''   # e.g. 's3://my-bucket/firepath-results/'
if S3_OUT:
    !aws s3 cp $BASE/saved.zip $S3_OUT